Resultaten wegschrijven in genormaliseerd datamodel. Voorlopig gesimuleerd als sqlite. Connectie met LSVI databank of andere masterdata nog uit te klaren?

In [ ]:
import sqlite3
from datetime import datetime
import pandas as pd

from arcgis.gis import GIS

### Connectie naar databank

In [ ]:
# Aanmaken 
def init_database(db_path="lsvi_resultaten_slank.sqlite"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    # Tabel 1: WaarnemingEvent
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS waarneming_event (
        collectie_id TEXT PRIMARY KEY,
        global_id TEXT,
        tijdstip DATETIME,
        waarnemer TEXT,
        locatie_x REAL,
        locatie_y REAL,
        EPSG INTEGER,
        doel_habitattype TEXT,
        bwk_id TEXT,
        survey_versie TEXT,
        created_date DATETIME,
        last_edited_date DATETIME
    );
    """)
    
    # Tabel 2: Resultaat (Nu ook voor de losse soorten!)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS resultaat (
        resultaat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        collectie_id TEXT,
        voorwaarde_id INTEGER,
        vraag_id TEXT,
        vraag_label TEXT,
        waarde_tekst TEXT,
        waarde_numeriek REAL,
        taxongroep_id INTEGER,
        FOREIGN KEY (collectie_id) REFERENCES waarneming_event(collectie_id) ON DELETE CASCADE
    );
    """)
    conn.commit()
    return conn

# Helperfunctie om ArcGIS Epoch Milliseconds om te zetten naar een ISO string
def format_agol_date(epoch_ms):
    if epoch_ms is None:
        return None
    # AGOL timestamps zijn in milliseconden, Python verwacht seconden
    return datetime.fromtimestamp(epoch_ms / 1000.0).strftime('%Y-%m-%d %H:%M:%S')

def run_etl(agol_url, username, password, feature_layer_item_id, db_path="lsvi_resultaten_slank.sqlite"):
    
    print(f"Verbinden met {agol_url}...")
    gis = GIS(agol_url, username, password)
    # Haal de Feature Layer op en lees de data volledig uit
    print(f"Feature layer {feature_layer_item_id} downloaden...")
    layer_item = gis.content.get(feature_layer_item_id)
    feature_layer = layer_item.layers[0]  

    # Vraag alle records op (1=1) inclusief WGS84 geometrie (out_sr=4326)
    features_result = feature_layer.query(where="1=1", out_sr=4326)
    print(f"Succesvol {len(features_result.features)} features gedownload. Start verwerking...")

    conn = init_database(db_path)
    cursor = conn.cursor()

    # 4. Loop door ELKE feature uit de layer
    for feature in features_result.features:
        geom = feature.geometry if feature.geometry else {}
        attrs = feature.attributes if feature.attributes else {}
        
        # Controleer of de cruciale collectie_id aanwezig is
        collectie_id = attrs.get('collectie_id')
        if not collectie_id:
            continue
            
        # Metadata extraheren en parsen
        global_id = attrs.get('globalid')
        waarnemer = attrs.get('last_edited_user')
        habitat_keuze = attrs.get('habitat_keuze')
        bwk_id = attrs.get('bwk_id')
        
        created_date_txt = format_agol_date(attrs.get('created_date'))
        last_edited_date_txt = format_agol_date(attrs.get('last_edited_date'))
        
        # Tijdstip van het bezoek bepalen (Datum-veld + Uur-veld combineren)
        datum_txt = format_agol_date(attrs.get('datum'))
        uur_txt = attrs.get('uur')  # string zoals "14:59"
        tijdstip_waarneming = f"{datum_txt.split(' ')[0]} {uur_txt}" if datum_txt and uur_txt else datum_txt

        # Geometrie
        x = geom.get('x')
        y = geom.get('y')
        epsg = geom.get('spatialReference', {}).get('wkid')

        # WaarnemingEvent wegschrijven
        cursor.execute("""
            INSERT INTO waarneming_event (
                collectie_id, global_id, tijdstip, waarnemer, locatie_X, locatie_Y, epsg, doel_habitattype, bwk_id, created_date, last_edited_date
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(collectie_id) DO UPDATE SET
                global_id=excluded.global_id,
                tijdstip=excluded.tijdstip,
                waarnemer=excluded.waarnemer,
                locatie_X=excluded.locatie_X,
                locatie_Y=excluded.locatie_Y,
                epsg=excluded.epsg,
                doel_habitattype=excluded.doel_habitattype,
                bwk_id=excluded.bwk_id,
                created_date=excluded.created_date,
                last_edited_date=excluded.last_edited_date;
            """, (collectie_id, global_id, tijdstip_waarneming, waarnemer, x, y, epsg, habitat_keuze, bwk_id, created_date_txt, last_edited_date_txt))
        
        # Resultaten wegschrijven
        # Eerst collectieID wissen voor overschrijven
        cursor.execute("DELETE FROM resultaat WHERE collectie_id = ?", (collectie_id,))

        # Loop dynamisch door alle kolommen van de feature
        for key, value in attrs.items():
            # Sla lege cellen, systeemenmerken en layout-hulpmiddelen over
            if value is None or str(value).strip() == "" or str(value).lower() == 'none':
                continue
            if key in ['objectid', 'globalid', 'collectie_id', 'datum', 'uur', 'waarnemer', 'habitat_keuze', 'bwk_id', 'created_date', 'created_user', 'last_edited_date', 'last_edited_user']:
                continue
            if key.startswith('note_lijst_'):
                continue
                
            # Extraheer het getal (VoorwaardeID) als de kolom begint met 'vraag_'
            voorwaarde_id = None
            if key.startswith("vraag_"):
                schone_id = key.replace("vraag_", "")
                if schone_id.isdigit():
                    voorwaarde_id = int(schone_id)
            
            # Controleer of het antwoord een select_multiple soortenlijst is (komma-gescheiden string)
            if isinstance(value, str) and "," in value:
                soorten = [s.strip() for s in value.split(",")]
                for soort in soorten:
                    if soort:
                        cursor.execute("""
                        INSERT INTO resultaat (collectie_id, voorwaarde_id, vraag_naam, waarde_tekst, waarde_numeriek)
                        VALUES (?, ?, ?, ?, NULL);
                        """, (collectie_id, voorwaarde_id, key, soort))
            else:
                # Normale single-choice vraag of tekstveld
                waarde_numeriek = None
                try:
                    waarde_numeriek = float(value)
                except ValueError:
                    pass  # Tekstwaarde, dus numeriek blijft NULL
                    
                cursor.execute("""
                INSERT INTO resultaat (collectie_id, voorwaarde_id, vraag_naam, waarde_tekst, waarde_numeriek)
                VALUES (?, ?, ?, ?, ?);
                """, (collectie_id, voorwaarde_id, key, str(value), waarde_numeriek))

    # Sluit alles netjes af
    conn.commit()
    conn.close()
    print("ETL afgerond. De SQLite database is volledig up-to-date.")

### Run ETL

In [ ]:
# Uitvoering
# Joost to do: credentials uit key vault of environment
agol_url = "https://gisservices.inbo.be/portal"
username = "joost.neujens@inbo.be"
password = "!JNE@g1m!"
feature_layer_item_id = "f8866108438b44cf89e9c0c31cc18c1c"
db_path = "../output/lsvi_resultaten.sqlite"

# Start de volledige pipeline
run_etl(agol_url, username, password, feature_layer_item_id, db_path)

### Load data export

In [ ]:
sqlite_path = "../output/lsvi_resultaten.sqlite"
conn = sqlite3.connect(sqlite_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)
print("Tables in database:")
display(tables)

print("\nWaarnemingEvent sample:")
display(pd.read_sql_query("SELECT * FROM waarneming_event LIMIT 20;", conn))

print("\nResultaat sample:")
results = pd.read_sql_query("SELECT * FROM resultaat WHERE collectie_id = '2e5acf0e-cef5-4d29-ad06-96900a597a7b';", conn)
display(results)

conn.close()